# Imports

In [50]:
import sys
from pathlib import Path

# Get the current notebook's directory (using the current working directory)
NOTEBOOK_DIR = Path.cwd()

# Navigate up to the project root (assumes notebook is in ./notebooks)
ROOT_DIR = NOTEBOOK_DIR.parent

# Path to the 'src' directory
SRC_DIR = ROOT_DIR / "src"

# Add 'src' to Python path so we can import modules like 'data.load_data'
sys.path.append(str(SRC_DIR))


In [51]:
import numpy                 as np
import pandas                as pd
%matplotlib inline
import matplotlib.pyplot     as plt
import seaborn               as sns
import umap.umap_            as umap
import sklearn.preprocessing as pp
import networkx              as nx
import warnings
import re

from data.load_data             import load_ecommerce_data
from features.transform_columns import to_snake_case_columns

from yellowbrick.cluster        import KElbowVisualizer, SilhouetteVisualizer
from sklearn.metrics            import silhouette_score, silhouette_samples
from plotly                     import express as px

from sklearn.ensemble           import RandomForestRegressor
from scipy.cluster              import hierarchy as hc


warnings.filterwarnings('ignore')


# Load Dataset

In [53]:
# Load data
df_raw = load_ecommerce_data("../data/raw/Ecommerce.csv")

df_raw

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,29-Nov-16,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,29-Nov-16,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,29-Nov-16,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,29-Nov-16,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,29-Nov-16,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,7-Dec-17,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,7-Dec-17,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,7-Dec-17,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,7-Dec-17,4.15,12680.0,France


# <font color='blue'> 📊  1.0 Data Description

## <span style="color:blue">1.1</span> Rename Columns

In [54]:
df1 = df_raw.copy()

In [55]:
df1 = to_snake_case_columns(df1)
df1.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,29-Nov-16,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,29-Nov-16,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,29-Nov-16,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,29-Nov-16,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,29-Nov-16,3.39,17850.0,United Kingdom


## <span style="color:blue">1.2</span> Data Dimensions

In [56]:
print(f'Number of rows: {df1.shape[0]}')
print(f'Number of columns: {df1.shape[1]}')

Number of rows: 541909
Number of columns: 8


## <span style="color:blue">1.3</span> Data Types

In [57]:
df1.dtypes

invoice_no       object
stock_code       object
description      object
quantity          int64
invoice_date     object
unit_price      float64
customer_id     float64
country          object
dtype: object

## <span style="color:blue">1.4</span> Check NA

In [58]:
df1.isna().sum()

invoice_no           0
stock_code           0
description       1454
quantity             0
invoice_date         0
unit_price           0
customer_id     135080
country              0
dtype: int64

## <span style="color:blue">1.5</span> Replace NA

In [59]:
# Get NA records
df_missing = df1.loc[df1['customer_id'].isna(), :]

# Get NOT NA records
df_not_missing  = df1.loc[~df1['customer_id'].isna(), :]

In [60]:
# create auxiliary dataframe with missing customer_id
df_backup = pd.DataFrame(df_missing['invoice_no'].drop_duplicates())
df_backup['customer_id'] = np.arange(19000, 19000 + len(df_backup), 1)

# merge auxiliary dataframe with df_missing
df_missing.drop(columns=['customer_id'], inplace=True)
df_missing = df_missing.merge(df_backup, on='invoice_no', how='left')

# concatenate df_not_missing and df_missing
df1 = pd.concat([df_not_missing, df_missing], axis=0)

df1.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,29-Nov-16,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,29-Nov-16,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,29-Nov-16,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,29-Nov-16,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,29-Nov-16,3.39,17850.0,United Kingdom


In [61]:
df1.isna().sum()

invoice_no         0
stock_code         0
description     1454
quantity           0
invoice_date       0
unit_price         0
customer_id        0
country            0
dtype: int64

## <span style="color:blue">1.6</span> Change dtype

In [62]:
# invoice_date
df1['invoice_date'] = pd.to_datetime(df1['invoice_date'], format='%d-%b-%y')

# customer id
df1['customer_id'] = df1['customer_id'].astype(int)

df1.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2016-11-29,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2016-11-29,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2016-11-29,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2016-11-29,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2016-11-29,3.39,17850,United Kingdom


In [63]:
df1.dtypes

invoice_no              object
stock_code              object
description             object
quantity                 int64
invoice_date    datetime64[ns]
unit_price             float64
customer_id              int64
country                 object
dtype: object

## <span style="color:blue">1.7</span> Descriptive Statistics

In [64]:
num_attributes = df1.select_dtypes(include=['int64', 'float64'])
cat_attributes = df1.select_dtypes(exclude=['int64', 'float64', 'datetime64[ns]'])

### <span style="color:blue">1.7.1</span> Numerical Attributes

In [65]:
# Calculate all desired statistics at once (central and dispersion)
stats = pd.DataFrame({
    'mean': num_attributes.mean(),
    'median': num_attributes.median(),
    'std': num_attributes.std(),
    'min': num_attributes.min(),
    'max': num_attributes.max(),
    'range': num_attributes.max() - num_attributes.min(),
    'skew': num_attributes.skew(),
    'kurtosis': num_attributes.kurtosis()
})

# Display the statistics
print("Numerical Attributes Statistics:")
stats

Numerical Attributes Statistics:


,mean,median,std,min,max,range,skew,kurtosis
quantity,9.552250,3.00,218.081158,-80995.00,80995.0,161990.00,-0.264076,119769.160031
unit_price,4.611114,2.08,96.759853,-11062.06,38970.0,50032.06,186.506972,59005.719097
customer_id,16688.840453,16249.00,2911.411352,12346.00,22709.0,10363.00,0.487449,-0.804287


### <span style="color:blue">1.7.2</span> Categorical Attributes

In [66]:
cat_attributes.head()

,invoice_no,stock_code,description,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,United Kingdom
1,536365,71053,WHITE METAL LANTERN,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,United Kingdom


In this first cycle, the Country column will not be used.

# <font color='blue'> 🧹2.0 Data Filtering

In [67]:
df2 = df1.copy()

In [68]:
df2.dtypes

invoice_no              object
stock_code              object
description             object
quantity                 int64
invoice_date    datetime64[ns]
unit_price             float64
customer_id              int64
country                 object
dtype: object

In [69]:

# === Numerical attributes ===
df2 = df2.loc[df2['unit_price'] >= 0.04, :]

# === Categorical attributes ====
df2 = df2[~df2['stock_code'].isin( ['POST', 'D', 'DOT', 'M', 'S', 'AMAZONFEE', 'm', 'DCGSSBOY', 'DCGSSGIRL', 'PADS', 'B', 'CRUK'] ) ]

# Remove column description
df2.drop(columns=['description'], inplace=True)

# country
df2 = df2[~df2['country'].isin(['European Community', 'Unspecified'])]

# quantity
df2_returns = df2.loc[df2['quantity'] < 0, :]
df2_purchases = df2.loc[df2['quantity'] > 0, :]


# <font color='blue'> ⚙️ 3.0 Feature Engineering

In [70]:
df3= df2.copy()

In [71]:
df3.shape

(536143, 7)

## <span style="color:blue">3.1</span> Feature Creation

In [72]:
# data reference
df_ref = (
    df3.drop( ['invoice_no', 'stock_code', 'quantity', 'invoice_date', 'unit_price', 'country'], axis=1 )
    .drop_duplicates( ignore_index=True )
)

### <span style="color:blue">3.1.1 </span> Gross Revenue



In [73]:
# Gross Revenue - Quantity * Unit Price
df2_purchases.loc[:, 'gross_revenue'] = df2_purchases.loc[:, 'quantity'] * df2_purchases.loc[:, 'unit_price']

# Monetary
df_monetary = df2_purchases.loc[:, ['customer_id', 'gross_revenue']].groupby( 'customer_id' ).sum().reset_index()
df_ref = pd.merge( df_ref, df_monetary, on='customer_id', how='left' )
df_ref.isna().sum()

customer_id       0
gross_revenue    91
dtype: int64

### <span style="color:blue">3.1.2</span> Recency - Day from Last Purchase

In [74]:
# Recency - Last Day Purchase
df_recency = (
    df2_purchases[['customer_id', 'invoice_date']]
    .groupby('customer_id').max().reset_index()
    .rename(columns={'invoice_date': 'last_day_purchase'})
)

df_recency['recency_days'] = (df2_purchases['invoice_date'].max() - df_recency['last_day_purchase']).dt.days
df_ref = pd.merge(df_ref, df_recency[['customer_id', 'recency_days']], on='customer_id', how='left')
df_ref.isna().sum()

customer_id       0
gross_revenue    91
recency_days     91
dtype: int64

### <span style="color:blue">3.1.3</span> Quantity of Purchases

In [75]:
# Frequency - Number of Purchases
df_frequency = df2_purchases[['customer_id', 'invoice_no']]. \
                  drop_duplicates().groupby('customer_id').\
                  count().reset_index().rename(columns={'invoice_no': 'qtde_invoices'})

df_ref = pd.merge(df_ref, df_frequency, on='customer_id', how='left')
df_ref.isna().sum()

customer_id       0
gross_revenue    91
recency_days     91
qtde_invoices    91
dtype: int64

### <span style="color:blue">3.1.4</span> Quantity Total of Items Purchased

In [76]:
# Number of Products Purchased
df_frequency = df2_purchases[['customer_id', 'quantity']].\
                    groupby('customer_id').sum().\
                    reset_index().rename(columns={'quantity': 'qtde_items'})

df_ref = pd.merge(df_ref, df_frequency, on='customer_id', how='left')
df_ref.isna().sum()

customer_id       0
gross_revenue    91
recency_days     91
qtde_invoices    91
qtde_items       91
dtype: int64

### <span style="color:blue">3.1.5</span> Quantity of products purchased 

In [77]:
# Numero de produtos
df_freq = df2_purchases[['customer_id', 'stock_code']].\
            groupby( 'customer_id' ).count().\
            reset_index().\
            rename( columns={'stock_code': 'qtde_products'} )

df_ref = pd.merge( df_ref, df_freq, on='customer_id', how='left' )
df_ref.isna().sum()

customer_id       0
gross_revenue    91
recency_days     91
qtde_invoices    91
qtde_items       91
qtde_products    91
dtype: int64

### <span style="color:blue">3.1.6</span> Avg Ticket

In [78]:
# Avg Ticket -  Gross Revenue / Frequency
df_avg_ticket = (
    df2_purchases[['customer_id', 'gross_revenue']]
    .groupby('customer_id').mean().reset_index().rename(columns={'gross_revenue': 'avg_ticket'})
)
df_ref = pd.merge(df_ref, df_avg_ticket, on='customer_id', how='left')
df_ref.isna().sum()

customer_id       0
gross_revenue    91
recency_days     91
qtde_invoices    91
qtde_items       91
qtde_products    91
avg_ticket       91
dtype: int64

### <span style="color:blue">3.1.7</span> Average Recency Days

In [79]:
df_aux = df2[['customer_id', 'invoice_date']].drop_duplicates().sort_values( ['customer_id', 'invoice_date'], ascending=[False, False] )
df_aux['previous_customer_id'] = df_aux['customer_id'].shift(-1) # next customer
df_aux['previous_date'] = df_aux['invoice_date'].shift(-1) # next invoince date

df_aux['avg_recency_days'] = df_aux.apply( lambda x: ( x['invoice_date'] - x['previous_date'] ).days if x['customer_id'] == x['previous_customer_id'] else np.nan, axis=1 )

df_aux = df_aux.drop( ['invoice_date', 'previous_customer_id', 'previous_date'], axis=1 ).dropna()

# average recency 
df_avg_recency_days = df_aux.groupby( 'customer_id' ).mean().reset_index()

# merge
df_ref = pd.merge( df_ref, df_avg_recency_days, on='customer_id', how='left' )
df_ref.isna().sum()

customer_id            0
gross_revenue         91
recency_days          91
qtde_invoices         91
qtde_items            91
qtde_products         91
avg_ticket            91
avg_recency_days    2816
dtype: int64

### <span style="color:blue">3.1.8</span> Purchase Frequency

In [80]:
# Step 1: Create an auxiliary DataFrame with purchase activity summary for each customer
df_aux = (
    df2_purchases[['customer_id', 'invoice_no', 'invoice_date']]  # Select relevant columns
    .drop_duplicates()  # Remove duplicate purchase records
    .groupby('customer_id')  # Group by customer
    .agg(
        max_=('invoice_date', 'max'),  # Last purchase date
        min_=('invoice_date', 'min'),  # First purchase date
        days_=('invoice_date', lambda x: (x.max() - x.min()).days + 1),  # Total active days
        buy_=('invoice_no', 'count')  # Total number of purchases
    )
    .reset_index()
)

# Step 2: Calculate frequency of purchases (buys per day)
df_aux['frequency'] = df_aux.apply(
    lambda row: row['buy_'] / row['days_'] if row['days_'] != 0 else 0,
    axis=1
)

# Step 3: Merge frequency information into the reference DataFrame
df_ref = pd.merge(
    df_ref,
    df_aux[['customer_id', 'frequency']],
    on='customer_id',
    how='left'  # Keep all rows from df_ref
)

# Step 4: Check for missing values after the merge
df_ref.isna().sum()

customer_id            0
gross_revenue         91
recency_days          91
qtde_invoices         91
qtde_items            91
qtde_products         91
avg_ticket            91
avg_recency_days    2816
frequency             91
dtype: int64

### <span style="color:blue">3.1.9</span> Returns

In [81]:
# Calculate total number of returned items per customer
df_returns = (
    df2_returns[['customer_id', 'quantity']]
    .groupby('customer_id')
    .sum()
    .reset_index()
    .rename(columns={'quantity': 'qtde_returns'})
)

# Convert return quantities to positive values
df_returns['qtde_returns'] = df_returns['qtde_returns'].abs()

# Merge return data into the reference DataFrame
df_ref = pd.merge(df_ref, df_returns, on='customer_id', how='left')

# Fill missing return quantities with 0 (i.e., no returns)
df_ref['qtde_returns'] = df_ref['qtde_returns'].fillna(0)

# Check for any remaining missing values in the DataFrame
df_ref.isna().sum()

customer_id            0
gross_revenue         91
recency_days          91
qtde_invoices         91
qtde_items            91
qtde_products         91
avg_ticket            91
avg_recency_days    2816
frequency             91
qtde_returns           0
dtype: int64

### <span style="color:blue">3.2.0</span> Basket Size - Average Quantity of Items per Basket ( Quantity )

- Invoice No = purchase
- Stock Code = Product
- Quantity = Item

In [82]:
df_aux =(df2_purchases.loc[:, ['customer_id', 'invoice_no', 'quantity']].groupby( 'customer_id' )
                                                                            .agg( n_purchase=( 'invoice_no', 'nunique'),
                                                                                  n_products=( 'quantity', 'sum' ) )
                                                                            .reset_index())
# calculation
df_aux['avg_basket_size'] = df_aux['n_products'] / df_aux['n_purchase']

# merge
df_ref = pd.merge( df_ref, df_aux[['customer_id', 'avg_basket_size']], how='left', on='customer_id' )
df_ref.isna().sum()

customer_id            0
gross_revenue         91
recency_days          91
qtde_invoices         91
qtde_items            91
qtde_products         91
avg_ticket            91
avg_recency_days    2816
frequency             91
qtde_returns           0
avg_basket_size       91
dtype: int64

### <span style="color:blue">3.2.1</span> Unique Basket Size - Average Quantity of distinct products per purchase

In [83]:
df_aux = ( df2_purchases.loc[:, ['customer_id', 'invoice_no', 'stock_code']].groupby( 'customer_id' )
                                                                            .agg( n_purchase=( 'invoice_no', 'nunique'),
                                                                                   n_products=( 'stock_code', 'nunique' ) )
                                                                            .reset_index() )

# calculation
df_aux['avg_unique_basket_size'] = df_aux['n_products'] / df_aux['n_purchase']

# merge
df_ref = pd.merge( df_ref, df_aux[['customer_id', 'avg_unique_basket_size']], how='left', on='customer_id' )
df_ref.isna().sum()

customer_id                  0
gross_revenue               91
recency_days                91
qtde_invoices               91
qtde_items                  91
qtde_products               91
avg_ticket                  91
avg_recency_days          2816
frequency                   91
qtde_returns                 0
avg_basket_size             91
avg_unique_basket_size      91
dtype: int64

# <font color='blue'> 📈 4.0 Exploratory Data Analysis

In [84]:
df4 = df_ref.dropna().copy()

# <font color='blue'> 🎯 5.0 Feature Spotlight: Selecting the Best Predictors

In [85]:
cols_selected = ['customer_id', 'gross_revenue', 'recency_days', 'qtde_products', 'frequency', 'qtde_returns'] 

df5 = df4[cols_selected].copy()
df5.head()

,customer_id,gross_revenue,recency_days,qtde_products,frequency,qtde_returns
0,17850,5391.21,372.0,297.0,17.000000,40.0
1,13047,3232.59,56.0,171.0,0.028302,35.0
2,12583,6705.38,2.0,232.0,0.040323,50.0
3,13748,948.25,95.0,28.0,0.017921,0.0
4,15100,876.00,333.0,3.0,0.073171,22.0


# <font color='blue'> 🛠️ 6.0 Data Transforming

In [86]:
df6 = df5.copy()

In [87]:
# Initialize MinMaxScalers
mm_gross_revenue = pp.MinMaxScaler()
mm_recency_days = pp.MinMaxScaler()
mm_qtde_products = pp.MinMaxScaler()
mm_frequency = pp.MinMaxScaler()
mm_qtde_returns = pp.MinMaxScaler()

# Apply MinMaxScaler only on selected numerical columns
df6['gross_revenue'] = mm_gross_revenue.fit_transform(df6[['gross_revenue']])
df6['recency_days'] = mm_recency_days.fit_transform(df6[['recency_days']])
df6['qtde_products'] = mm_qtde_products.fit_transform(df6[['qtde_products']])
df6['frequency'] = mm_frequency.fit_transform(df6[['frequency']])
df6['qtde_returns'] = mm_qtde_returns.fit_transform(df6[['qtde_returns']])

cols_selected = ['customer_id', 'gross_revenue', 'recency_days', 'qtde_products', 'frequency', 'qtde_returns'] 

# Verificar resultado
df6[cols_selected].head()

,customer_id,gross_revenue,recency_days,qtde_products,frequency,qtde_returns
0,17850,0.019292,0.997319,0.037770,1.000000,0.000494
1,13047,0.011559,0.150134,0.021692,0.001345,0.000432
2,12583,0.024000,0.005362,0.029476,0.002052,0.000617
3,13748,0.003375,0.254692,0.003445,0.000734,0.000000
4,15100,0.003116,0.892761,0.000255,0.003985,0.000272


# <font color='blue'> 🎯 7.0 Study of the Space of Embeddings

In [88]:
df7 = df6.copy()

In [89]:
# X receives all columns from df6 except 'customer_id'
X = df7.drop(columns=['customer_id']).copy()

# Check result
X.head()

,gross_revenue,recency_days,qtde_products,frequency,qtde_returns
0,0.019292,0.997319,0.037770,1.000000,0.000494
1,0.011559,0.150134,0.021692,0.001345,0.000432
2,0.024000,0.005362,0.029476,0.002052,0.000617
3,0.003375,0.254692,0.003445,0.000734,0.000000
4,0.003116,0.892761,0.000255,0.003985,0.000272


### 7.4 Tree-Based Method

In [90]:
# training dataset
X = df7.drop(columns=['customer_id', 'gross_revenue'], axis=1)
y = df7['gross_revenue']

# model definition
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# model training
rf_model.fit(X, y)

# Leaf
df_leaf = pd.DataFrame(rf_model.apply(X))

# Fit UMAP and transform the data
reducer = umap.UMAP(random_state=42)       # Initialize UMAP with fixed random seed
embedding = reducer.fit_transform(df_leaf)       # Fit UMAP on the data and transform it into 2D

# Create a new DataFrame with the UMAP embeddings
df_umap_tree = pd.DataFrame({
    'embedding_x': embedding[:, 0],        # First UMAP component (x-axis)
    'embedding_y': embedding[:, 1]         # Second UMAP component (y-axis)
})


# <font color='blue'> 🚀 9.0 Model Training

In [98]:
X = df_umap_tree.copy()

## <span style="color:blue">9.1</span>  Hierarchical Clustering

In [103]:
# Define the number of clusters
k = 10

# Model definition (using Ward linkage as an example — can be 'average', 'complete', etc.)
hc_model = hc.linkage(X, method='ward')

# Cut the dendrogram to form exactly k clusters
labels = hc.fcluster(hc_model, k, criterion='maxclust')

# Check unique clusters
print(np.unique(labels))

[ 1  2  3  4  5  6  7  8  9 10]


## <span style="color:blue">9.2</span> Cluster Validation

In [104]:
## WSS (Within-cluster sum of squares)
# print('WSS value: ', model_kmeans.inertia_)

## Silhouette Score
ss = silhouette_score(X, labels, metric='euclidean')
print(f"Silhouette Score (SS): {ss:.3f}")

Silhouette Score (SS): 0.681


# <font color='blue'> 🌐 10.0 Cluster Analysis: Unveiling Hidden Patterns

In [105]:
df10 = df5.copy()
df10['cluster'] = labels
df10.head()

,customer_id,gross_revenue,recency_days,qtde_products,frequency,qtde_returns,cluster
0,17850,5391.21,372.0,297.0,17.000000,40.0,10
1,13047,3232.59,56.0,171.0,0.028302,35.0,9
2,12583,6705.38,2.0,232.0,0.040323,50.0,10
3,13748,948.25,95.0,28.0,0.017921,0.0,1
4,15100,876.00,333.0,3.0,0.073171,22.0,7


## <span style="color:blue">10.2</span> Cluster Profile

In [106]:
# Number os customers
df_cluster = df10[['cluster', 'customer_id']].groupby('cluster').count().reset_index().rename(columns={'customer_id': 'count_customer'})
df_cluster['perc_customer'] = df_cluster['count_customer'] / df_cluster['count_customer'].sum() * 100

# Avg Gross Revenue
df_cluster['avg_gross_revenue'] = df10[['cluster', 'gross_revenue']].groupby('cluster').mean().reset_index()['gross_revenue']

# Avg Recency Days
df_cluster['avg_recency_days'] = df10[['cluster', 'recency_days']].groupby('cluster').mean().reset_index()['recency_days']

# Avg Quantity of Products
df_cluster['avg_qtde_products'] = df10[['cluster', 'qtde_products']].groupby('cluster').mean().reset_index()['qtde_products']

# Avg Frequency
df_cluster['avg_frequency'] = df10[['cluster', 'frequency']].groupby('cluster').mean().reset_index()['frequency']

# Avg Returns
df_cluster['avg_returns'] = df10[['cluster', 'qtde_returns']].groupby('cluster').mean().reset_index()['qtde_returns']


# Store old cluster labels
df_cluster['old_cluster'] = df_cluster['cluster']

# Sort clusters by avg_gross_revenue (descending) and reassign cluster labels
df_cluster = (
    df_cluster
    .sort_values('avg_gross_revenue', ascending=False)
    .reset_index(drop=True)
)

# Assign cluster labels starting from 1
df_cluster['cluster'] = df_cluster.index + 1

# Create mapping dictionary
cluster_map = dict(zip(df_cluster['old_cluster'], df_cluster['cluster']))

# Apply mapping safely
df10['cluster_new'] = df10['cluster'].map(cluster_map)

# Check for unmapped clusters
print("Unmapped clusters:")
print(df10[df10['cluster_new'].isna()]['cluster'].unique())

# Replace old cluster
df10['cluster'] = df10['cluster_new']
df10.drop(columns=['cluster_new'], inplace=True)

# Drop helper column
df_cluster.drop(columns=['old_cluster'], inplace=True)

df_cluster


Unmapped clusters:
[]


,cluster,count_customer,perc_customer,avg_gross_revenue,avg_recency_days,avg_qtde_products,avg_frequency,avg_returns
0,1,469,15.796564,9176.512431,21.392324,423.236674,0.094390,321.650320
1,2,130,4.378579,4689.930154,47.407692,103.900000,0.057342,20.753846
2,3,145,4.883799,3164.253379,34.427586,173.875862,0.060415,23.482759
3,4,355,11.956888,2419.750366,44.219718,128.146479,0.042380,21.580282
4,5,407,13.708319,1693.116388,54.936118,89.334152,0.049822,11.496314
5,6,370,12.462108,1218.609865,60.548649,54.348649,0.043822,7.221622
6,7,322,10.845402,971.025435,75.329193,37.462733,0.074586,6.571429
7,8,162,5.456383,819.306358,85.728395,27.944444,0.124034,4.000000
8,9,253,8.521388,650.767905,45.003953,13.474308,0.029598,0.869565
9,10,356,11.990569,509.569298,167.674157,14.584270,0.489454,26.941011


### <span style="color:blue">10.2.1</span> Customer Segmentation Results 


#### 📊 Cluster Profiling (10 Clusters)

Below is the corrected and aligned interpretation of each cluster based on the table provided.

---

##### 🥇 **Cluster 1 – Insiders (Top Value Customers)**
- **Represent:** 15.8% of customers  
- **Highest average revenue:** ≈ 9,176  
- **Very recent activity:** ≈ 21 days  
- **Extremely high volume:** ≈ 423 products  
- **Very high returns:** ≈ 322 (expected due to heavy purchasing volume)  

**Profile:**  
Top-tier clients driving a large share of total revenue. Very active, highly engaged, and extremely valuable despite operational cost from returns.

**➡ Core Insiders — Top Priority Segment**

---

##### 🥈 **Cluster 2 – High-Value, Low-Frequency Buyers**
- **Represent:** 4.4% of customers  
- **High revenue:** ≈ 4,690  
- **Medium recency:** ≈ 47 days  
- **Moderate volume:** ≈ 104 products  
- **Moderate returns:** ≈ 21  

**Profile:**  
High spenders with lower purchase frequency. Strong candidates for reactivation and loyalty programs.

**➡ Secondary Insiders — High Value, Lower Engagement**

---

##### 🥉 **Cluster 3 – Loyal Mid-High Buyers**
- **Represent:** 4.9% of customers  
- **Revenue:** ≈ 3,164  
- **Recency:** ≈ 34 days  
- **High volume:** ≈ 174 products  
- **Low returns:** ≈ 23  

**Profile:**  
Consistent and profitable customers with solid engagement and low operational cost.

**➡ Strong mid-tier with upgrade potential**

---

##### 🔹 **Cluster 4 – Stable Mid-Tier Buyers**
- **Represent:** 12.0% of customers  
- **Revenue:** ≈ 2,420  
- **Recency:** ≈ 44 days  
- **Volume:** ≈ 128 products  
- **Low returns:** ≈ 22  

**Profile:**  
Reliable and predictable customers. Good retention and stable revenue contribution.

**➡ Retain with loyalty programs**

---

##### 🔹 **Cluster 5 – Medium Value, Low Engagement**
- **Represent:** 13.7% of customers  
- **Revenue:** ≈ 1,693  
- **Recency:** ≈ 55 days  
- **Volume:** ≈ 89 products  
- **Low returns:** ≈ 11  

**Profile:**  
Balanced and low-maintenance group. Moderate engagement and profitability.

**➡ Solid base customers**

---

##### 🔹 **Cluster 6 – Low-Mid Value Buyers**
- **Represent:** 12.5% of customers  
- **Revenue:** ≈ 1,219  
- **Recency:** ≈ 61 days  
- **Volume:** ≈ 54 products  
- **Low returns:** ≈ 7  

**Profile:**  
Low frequency and low spend, but not operationally costly.

**➡ Reactivation opportunity**

---

##### 🔹 **Cluster 7 – Low Spenders, Aging Recency**
- **Represent:** 10.8% of customers  
- **Revenue:** ≈ 971  
- **Recency:** ≈ 75 days  
- **Volume:** ≈ 37 products  
- **Low returns:** ≈ 6.6  

**Profile:**  
Low-value and increasingly disengaged customers.

**➡ Low priority segment**

---

##### 🔹 **Cluster 8 – Very Low Engagement Buyers**
- **Represent:** 5.5% of customers  
- **Revenue:** ≈ 819  
- **Recency:** ≈ 86 days  
- **Volume:** ≈ 28 products  
- **Very low returns:** ≈ 4  

**Profile:**  
Low engagement, low spend, but operationally cheap.

**➡ Passive customers**

---

##### ⚠️ **Cluster 9 – Low Spenders, Almost Inactive**
- **Represent:** 8.5% of customers  
- **Revenue:** ≈ 651  
- **Recency:** ≈ 45 days  
- **Very low volume:** ≈ 13 products  
- **Minimal returns:** ≈ 0.9  

**Profile:**  
Very low contribution segment. Almost inactive buyers.

**➡ Churn risk / low priority**

---

##### ❌ **Cluster 10 – High Recency, High Return Risk**
- **Represent:** 12.0% of customers  
- **Lowest revenue:** ≈ 510  
- **Very long recency:** ≈ 168 days  
- **Very low volume:** ≈ 15 products  
- **Extremely high returns:** ≈ 27  

**Profile:**  
Low value and high operational cost. Likely unprofitable segment.

**➡ Exclude from Insiders — Monitor for churn**

---

# <font color='blue'> ✨ 11.0 Deploy to Production

## 11.1 Insert Into SQLITE

In [113]:
# create table
import sqlite3

from sqlalchemy import create_engine

In [114]:
df10.dtypes

customer_id        int64
gross_revenue    float64
recency_days       int64
qtde_products      int64
frequency        float64
qtde_returns       int64
cluster            int64
dtype: object

In [115]:
# change dtypes
df10['recency_days'] = df10['recency_days'].astype(int)
df10['qtde_products'] = df10['qtde_products'].astype(int)
df10['qtde_returns'] = df10['qtde_returns'].astype(int)

In [116]:

query_create_table_insiders = """
CREATE TABLE IF NOT EXISTS insiders (
    customer_id       INTEGER,
    gross_revenue     FLOAT,
    recency_days      INTEGER,
    qtde_products     INTEGER,
    frequency         FLOAT,
    qtde_returns      INTEGER,
    cluster           INTEGER
)
    
"""

conn = sqlite3.connect('../data/processed/insiders_db.sqlite')
conn.execute(query_create_table_insiders)
conn.commit()

# insert data with sqlalhemy    
create_engine('sqlite:///../data/processed/insiders_db.sqlite')
df10.to_sql('insiders', con=conn, if_exists='replace', index=False)

2969

In [117]:
# consulting database
query = """ SELECT * FROM insiders; """
df_insiders = pd.read_sql(query, con=conn)

df_insiders.head()

df_insiders.shape

(2969, 7)